# 1. Introduction

In this upgraded version, I have improve our model’s performance by:

1. Applying a log transformation to SalePrice to make the target more normally distributed.

2. Using GridSearchCV to tune important XGBoost hyperparameters for optimal performance.

<h1>2. Import requireds Libraries</h1>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

from xgboost import XGBRegressor
# Warnings
import warnings
warnings.filterwarnings('ignore')


# 3. Load the Datasets

In [ ]:
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)


# 4. Quick Data Overview

In [ ]:
# Display sample data
train.head()


In [ ]:
# Basic information
train.info()


In [ ]:
# Check missing values
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing.head(15)


# 5. Data Cleaning

In [ ]:
for df in [train, test]:
    for col in df.columns:
        if df[col].dtype == "object":
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)



# 6. Feature Engineering

In [ ]:
import scipy.stats as stats

# Before transformation
sns.histplot(train['SalePrice'], kde=True)
plt.title("SalePrice Distribution (Before Log)")
plt.show()

# Apply log transformation
train["SalePrice"] = np.log1p(train["SalePrice"])

# After transformation
sns.histplot(train['SalePrice'], kde=True, color="green")
plt.title("SalePrice Distribution (After Log)")
plt.show()



# 8. Feature Encoding

In [ ]:
all_data = pd.concat([train.drop('SalePrice', axis=1), test], axis=0)

obj_cols = all_data.select_dtypes(include='object').columns
lbl = LabelEncoder()
for col in obj_cols:
    all_data[col] = lbl.fit_transform(all_data[col])

X = all_data.iloc[:train.shape[0], :]
X_test = all_data.iloc[train.shape[0]:, :]
y = train['SalePrice']


# 8. Train-Test Split

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)


# 9. Hyperparameter Tuning (Grid Search)

In [ ]:
xgb_model = XGBRegressor(objective='reg:squarederror', random_state=42)

param_grid = {
    'n_estimators': [1000, 2000],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 4, 5],
    'subsample': [0.8, 1],
    'colsample_bytree': [0.8, 1],
    'reg_alpha': [0, 0.1],
    'reg_lambda': [1, 1.5]
}

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print("✅ Best Parameters:", grid_search.best_params_)



# 10. Training with Best Parameters

In [ ]:
best_params = grid_search.best_params_

xgb_best = XGBRegressor(
    objective='reg:squarederror',
    **best_params,
    random_state=42
)

xgb_best.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric='rmse',
    verbose=False,
    early_stopping_rounds=100
)


# 11. Validation Performance

In [ ]:
preds = xgb_best.predict(X_valid)
rmse = np.sqrt(mean_squared_error(y_valid, preds))
print("Validation RMSE:", rmse)


# 12. Cross-Validation for Model Robustness

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = np.sqrt(-cross_val_score(xgb_best, X, y, scoring="neg_mean_squared_error", cv=kfold))
print("Cross-validation RMSE:", cv_scores.mean())


# 13. Feature Importance Visualization

In [ ]:
importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_best.feature_importances_
}).sort_values(by='Importance', ascending=False).head(15)

plt.figure(figsize=(10,6))
sns.barplot(data=importances, x='Importance', y='Feature', palette='viridis')
plt.title("Top 15 Important Features")
plt.show()


# 14. Final Predictions and Submission

In [ ]:
final_preds = np.expm1(xgb_best.predict(X_test))

submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": final_preds
})

submission.to_csv("submission.csv", index=False)
print("✅ submission.csv created successfully!")


# 15. Conclusion

✅ Applied log transformation for target normalization<br>
✅ Tuned XGBoost hyperparameters for improved accuracy<br>
✅ Visualized feature importance<br>
✅ Created submission file for Kaggle leaderboard